# BIST Temel Analiz ve Filtreleme

**Veri Kaynakları:**
- **Bilanço:** isyatirim.com.tr MaliTablo API (doğrudan HTTP)
- **Sektör / Temettü:** borsapy
- **Fiyat / Teknik / Makro:** tvDatafeed
- **BIST Evren Listesi:** tradingview-screener

**Metodoloji:** 5 sektör grubu (Sınai, Finansal, NAV, Emtia, Altyapı) her biri kendi değerleme modeliyle.  
**TR Kalibrasyonu:** Altman Z eşik=1.0, F-Score eşik=4, TÜFE=%35, trend veto eşikleri.

In [ ]:
# ── Kurulum ──────────────────────────────────────────────────────────────────
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip("borsapy", "tvdatafeed", "tradingview-screener", "requests",
    "pandas", "numpy", "openpyxl", "ta", "tqdm")

# Google Drive bağlantısı (Colab)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive/BIST_v9"
    IN_COLAB = True
except Exception:
    DRIVE_ROOT = "/tmp/BIST_v9"
    IN_COLAB = False

import os
os.makedirs(f"{DRIVE_ROOT}/cache", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/raporlar", exist_ok=True)
print(f"Drive root: {DRIVE_ROOT}  |  Colab: {IN_COLAB}")

In [ ]:
# ── Import'lar ve Konfigürasyon ───────────────────────────────────────────────
import requests, pickle, time, warnings, json
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

# ── TR Kalibrasyon Sabitleri ─────────────────────────────────────────────────
TUFE          = 0.35        # yıllık TÜFE tahmini
RISK_FREE     = 0.40        # TR 10Y tahvil yaklaşık
ERP           = 0.06        # Equity risk premium
WACC_DEFAULT  = 0.28        # sınai şirketler için başlangıç WACC
ALTMAN_VETO   = 1.0         # Z < bu değer → kırmızı alarm
FSCORE_MIN    = 4           # F-Score minimumu
CAR_MIN       = 0.055       # Banka leverage min (%5.5)
NPL_MAX       = 0.08        # Banka takipteki kredi maks
PROFIT_DROP_VETO   = -0.40  # Kâr düşüşü veto eşiği (sınai/nav)
REVENUE_DROP_VETO  = -0.30  # Hasılat düşüşü veto eşiği
BANK_PROFIT_DROP   = -0.60  # Banka kâr veto (daha yumuşak)

# Veto kapalı sektörler (volatil doğa)
VETO_OFF_SECTORS = {
    "Rafineri", "Madencilik", "Demir_Celik", "Elektrik_Uretim",
    "Dogalgaz", "Telekom", "Havayolu", "Liman", "Lojistik", "Denizcilik"
}

# Özel ticker override
TICKER_OVERRIDE = {
    "KCHOL": "Holding", "SAHOL": "Holding", "DOHOL": "Holding", "AGHOL": "Holding",
    "ENDAE": "Elektrik_Uretim", "AKSEN": "Elektrik_Uretim", "ZOREN": "Elektrik_Uretim",
    "BULGS": "Yatirim_Ortakligi", "GOZDE": "Yatirim_Ortakligi",
    "FENER": "Spor", "GSRAY": "Spor", "BJKAS": "Spor", "TSPOR": "Spor",
    "ALTIN": "EMTIA_SERTIFIKASI", "GUMUS": "EMTIA_SERTIFIKASI",
}

# Niş kayıplar (bilanço API'de yok)
NIS_KAYIPLAR = {
    "TURSG", "ANSGR", "ANHYT", "AKGRT",
    "ISFIN", "LIDFA", "ULUFA", "VAKFA", "VAKFN",
    "DSTKF", "KTLEV", "GLCVY", "BRKVY", "ALBRK"
}

print("Konfigürasyon yüklendi.")

In [ ]:
# ── Yardımcı Fonksiyonlar ─────────────────────────────────────────────────────

def _tr_norm(s: str) -> str:
    """Türkçe lowercase bug'ını ve aksanlı harfleri normalize eder."""
    return (
        str(s).lower()
        .replace("\u0307", "")   # combining dot (İ→i sorunu)
        .replace("ı", "i").replace("ş", "s").replace("ç", "c")
        .replace("ğ", "g").replace("ü", "u").replace("ö", "o")
        .replace("İ", "i").replace("Ş", "s").replace("Ç", "c")
        .replace("Ğ", "g").replace("Ü", "u").replace("Ö", "o")
        .strip()
    )


def _cache_path(ticker: str) -> Path:
    return Path(DRIVE_ROOT) / "cache" / f"fin_{ticker}.pkl"


def _load_cache(ticker: str):
    p = _cache_path(ticker)
    if p.exists():
        age = time.time() - p.stat().st_mtime
        if age < 86400 * 7:   # 7 gün geçerliliği
            with open(p, "rb") as f:
                return pickle.load(f)
    return None


def _save_cache(ticker: str, data):
    with open(_cache_path(ticker), "wb") as f:
        pickle.dump(data, f)


def _find_item(df: pd.DataFrame, candidates: list[str]) -> pd.Series | None:
    """
    Bilanço DataFrame'inde kalem arar.
    df: index=kalem_adı, columns=dönem kolonları
    """
    norm_index = {_tr_norm(k): k for k in df.index}
    for cand in candidates:
        key = _tr_norm(cand)
        # tam eşleşme
        if key in norm_index:
            return df.loc[norm_index[key]]
        # içerir eşleşme
        for nk, orig in norm_index.items():
            if key in nk or nk in key:
                return df.loc[orig]
    return None


def safe_div(a, b, default=np.nan):
    try:
        if b == 0 or pd.isna(b):
            return default
        return a / b
    except Exception:
        return default


print("Yardımcı fonksiyonlar tanımlandı.")

In [ ]:
# ── isyatirim MaliTablo API — Bilanço Çekme ────────────────────────────────

MALI_TABLO_URL = (
    "https://www.isyatirim.com.tr/_layouts/15/IsYatirim.Website/"
    "Common/Data.aspx/MaliTablo"
)

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/javascript, */*",
    "Referer": "https://www.isyatirim.com.tr/",
}


def _fetch_mali_tablo_raw(
    ticker: str,
    financial_group: str = "XI_29",
    years: list[int] | None = None,
    periods: list[int] | None = None,
) -> dict | None:
    """
    isyatirim API'sine doğrudan HTTP çağrısı.
    financial_group: XI_29 (sınai), UFRS (banka), UFRS_K (sigorta/konsolide)
    """
    today = datetime.now()
    if years is None:
        years = [today.year - 1, today.year - 1, today.year, today.year]
    if periods is None:
        periods = [6, 12, 3, 6]  # son 4 dönem

    params = {
        "companyCode": ticker,
        "exchange": "TRY",
        "financialGroup": financial_group,
        "year1": years[0], "period1": periods[0],
        "year2": years[1], "period2": periods[1],
        "year3": years[2], "period3": periods[2],
        "year4": years[3], "period4": periods[3],
    }

    for attempt in range(3):
        try:
            resp = requests.get(
                MALI_TABLO_URL, params=params, headers=HEADERS,
                timeout=45
            )
            resp.raise_for_status()
            data = resp.json()
            if data.get("value"):
                return data
            return None
        except Exception as e:
            if attempt == 2:
                print(f"  [WARN] {ticker} MaliTablo çekme hatası: {e}")
                return None
            time.sleep(2 ** attempt)
    return None


def _parse_mali_tablo(raw: dict) -> pd.DataFrame:
    """
    API yanıtını DataFrame'e çevirir.
    Index: FINANCIAL_ITEM_NAME_TR
    Columns: "YYYY/M" formatında dönemler
    """
    rows = raw.get("value", [])
    if not rows:
        return pd.DataFrame()

    # Dönem kolonlarını tespit et
    period_cols = [k for k in rows[0].keys()
                   if k.startswith("PERIOD_") and k.endswith("_VALUE")]

    records = {}
    period_labels = {}

    for row in rows:
        name = row.get("FINANCIAL_ITEM_NAME_TR") or row.get("FINANCIAL_ITEM_CODE", "")
        vals = {}
        for pc in period_cols:
            # PERIOD_1_VALUE → dönem etiketi PERIOD_1_YEAR + PERIOD_1_MONTH
            idx = pc.replace("_VALUE", "")  # PERIOD_1
            yr  = row.get(f"{idx}_YEAR")
            mo  = row.get(f"{idx}_MONTH")
            if yr and mo:
                label = f"{yr}/{mo}"
                period_labels[pc] = label
                try:
                    vals[label] = float(row.get(pc) or 0)
                except (TypeError, ValueError):
                    vals[label] = np.nan
        records[name] = vals

    df = pd.DataFrame(records).T
    # Kolonları kronolojik sırala
    def _period_sort_key(col):
        try:
            y, m = col.split("/")
            return int(y) * 100 + int(m)
        except Exception:
            return 0
    df = df[sorted(df.columns, key=_period_sort_key)]
    return df


def get_balance_sheet(
    ticker: str,
    financial_group: str = "XI_29",
    use_cache: bool = True,
) -> pd.DataFrame:
    """
    Ticker için bilanço verisini çeker (cache destekli).
    """
    cache_key = f"{ticker}_{financial_group}"
    cached = _load_cache(cache_key) if use_cache else None
    if cached is not None:
        return cached

    today = datetime.now()
    cur_y, cur_m = today.year, today.month

    # Son 4 dönem: son çeyrek + 3 geriye
    # En son mevcut çeyrek tahmini
    if cur_m >= 11:
        last_q = (cur_y, 9)   # Q3 yayınlanmış olabilir
    elif cur_m >= 8:
        last_q = (cur_y, 6)
    elif cur_m >= 5:
        last_q = (cur_y, 3)
    else:
        last_q = (cur_y - 1, 12)

    # 4 dönem geriye
    def prev_quarter(y, m):
        qmap = {3: 12, 6: 3, 9: 6, 12: 9}
        nm = qmap[m]
        ny = y - 1 if nm == 12 and m == 3 else y
        return ny, nm

    d4 = last_q
    d3 = prev_quarter(*d4)
    d2 = prev_quarter(*d3)
    d1 = prev_quarter(*d2)

    raw = _fetch_mali_tablo_raw(
        ticker, financial_group,
        years   = [d1[0], d2[0], d3[0], d4[0]],
        periods = [d1[1], d2[1], d3[1], d4[1]],
    )
    if raw is None:
        return pd.DataFrame()

    df = _parse_mali_tablo(raw)
    if not df.empty:
        _save_cache(cache_key, df)
    return df


print("isyatirim MaliTablo modülü hazır.")

In [ ]:
# ── TTM (Trailing 12 Months) Hesaplama ────────────────────────────────────

def _parse_period(col: str) -> tuple[int, int]:
    """'2026/3' → (2026, 3)"""
    y, m = col.split("/")
    return int(y), int(m)


def get_ttm_value(df: pd.DataFrame, candidates: list[str]) -> float:
    """
    AKIŞ kalemi için TTM hesaplar.
    isyatirim kümülatif çalışır:
      - Son dönem Q4 ise yıllık direkt
      - Son dönem Qx ise: TTM = son_dönem + önceki_yıl_Q4 - önceki_yıl_same_Q
    """
    series = _find_item(df, candidates)
    if series is None:
        return np.nan

    cols = sorted(series.dropna().index, key=lambda c: _parse_period(c)[0]*100 + _parse_period(c)[1])
    if not cols:
        return np.nan

    last_col = cols[-1]
    last_y, last_m = _parse_period(last_col)

    # Q4 → yıllık, direkt kullan
    if last_m == 12:
        return float(series[last_col])

    # TTM = son_dönem + önceki_yıl_Q4 - önceki_yıl_same_quarter
    prev_annual_col = f"{last_y - 1}/12"
    prev_same_col   = f"{last_y - 1}/{last_m}"

    v_last       = float(series.get(last_col, np.nan))
    v_prev_ann   = float(series.get(prev_annual_col, np.nan))
    v_prev_same  = float(series.get(prev_same_col, np.nan))

    if any(np.isnan(x) for x in [v_last, v_prev_ann, v_prev_same]):
        # Yeterli dönem yoksa son mevcut değeri yıllıklaştır
        if not np.isnan(v_last) and last_m > 0:
            return v_last * 12 / last_m
        return np.nan

    return v_last + v_prev_ann - v_prev_same


def get_stock_value(df: pd.DataFrame, candidates: list[str]) -> float:
    """
    STOK kalemi için son dönem değerini döndürür (TTM yapma).
    """
    series = _find_item(df, candidates)
    if series is None:
        return np.nan
    cols = sorted(series.dropna().index, key=lambda c: _parse_period(c)[0]*100 + _parse_period(c)[1])
    if not cols:
        return np.nan
    return float(series[cols[-1]])


def get_prev_year_value(df: pd.DataFrame, candidates: list[str]) -> float:
    """
    Bir önceki yılın aynı dönem akış değerini döndürür (trend karşılaştırması için).
    """
    series = _find_item(df, candidates)
    if series is None:
        return np.nan
    cols = sorted(series.dropna().index, key=lambda c: _parse_period(c)[0]*100 + _parse_period(c)[1])
    if len(cols) < 2:
        return np.nan
    # İkinci en son dönem
    last_y, last_m = _parse_period(cols[-1])
    prev_col = f"{last_y - 1}/{last_m}"
    if prev_col in series.index:
        return float(series[prev_col])
    return float(series[cols[-2]])


print("TTM modülü hazır.")

In [ ]:
# ── Kalem Sözlükleri ──────────────────────────────────────────────────────

# Her kalem hem Türkçe hem İngilizce aday listesiyle
ITEMS_SINAI = {
    "varlik": [
        "toplam varlık", "aktif toplam", "toplam varliklar",
        "total assets"
    ],
    "ozkaynak": [
        "ana ortaklığa ait özkaynak", "toplam özkaynak", "özkaynaklar",
        "ana ortaklara ait ozkaynak",
        "stockholders equity", "common stock equity", "total equity"
    ],
    "hasilat": [
        "hasılat", "satış gelirleri", "net satışlar",
        "total revenue", "revenues"
    ],
    "net_kar": [
        "dönem net kâr", "dönem net kar", "net dönem kârı", "net kar",
        "net income", "net income common stockholders"
    ],
    "faaliyet_kar": [
        "esas faaliyet kârı", "faaliyet karı", "faaliyet kari",
        "ebit", "operating income"
    ],
    "brut_kar": [
        "brüt kâr", "brut kar", "satış karı",
        "gross profit"
    ],
    "cfo": [
        "işletme faaliyetlerinden nakit akışları",
        "isletme faaliyetlerinden nakit akislari",
        "operating cash flow", "cash from operations"
    ],
    "capex": [
        "maddi duran varlık alımları", "yatırım harcamaları",
        "capital expenditures", "capex"
    ],
    "amortisman": [
        "amortisman ve itfa", "amortisman",
        "depreciation", "depreciation amortization"
    ],
    "fin_borc_kv": [
        "kısa vadeli finansal borçlar", "kv finansal borc",
        "short term borrowings"
    ],
    "fin_borc_uv": [
        "uzun vadeli finansal borçlar", "uv finansal borc",
        "long term debt"
    ],
    "nakit": [
        "nakit ve nakit benzerleri", "nakit ve nakite esit varliklar",
        "cash and cash equivalents"
    ],
    "donen_varlik": [
        "dönen varlıklar", "donen varliklar",
        "current assets"
    ],
    "kv_yukum": [
        "kısa vadeli yükümlülükler", "kv yukumlulukler",
        "current liabilities"
    ],
    "uv_yukum": [
        "uzun vadeli yükümlülükler", "uv yukumlulukler",
        "long term liabilities"
    ],
    "faiz_gideri": [
        "finansman giderleri", "faiz giderleri",
        "interest expense", "finance costs"
    ],
}

ITEMS_BANKA = {
    "varlik": [
        "aktif toplam", "toplam aktifler",
        "total assets"
    ],
    "ozkaynak": [
        "özkaynaklar", "toplam özkaynak",
        "stockholders equity"
    ],
    "net_kar": [
        "dönem net kârı", "dönem net kar",
        "net income"
    ],
    "faiz_geliri": [
        "faiz gelirleri", "i. faiz gelirleri",
        "interest income"
    ],
    "faiz_gideri": [
        "faiz giderleri", "ii. faiz giderleri",
        "interest expense"
    ],
    "krediler": [
        "krediler", "kredi ve alacaklar",
        "loans", "net loans"
    ],
    "mevduat": [
        "mevduat", "toplam mevduat",
        "deposits"
    ],
    "takipteki": [
        "takipteki krediler", "donuk alacaklar",
        "non performing loans", "npl"
    ],
}

print("Kalem sözlükleri yüklendi.")

In [ ]:
# ── BIST Evren Listesi — tradingview-screener ─────────────────────────────

from tradingview_screener import Query, Column


def get_bist_universe(limit: int = 1500) -> pd.DataFrame:
    """
    BIST'teki tüm hisseleri çeker.
    Sütunlar: ticker, name, market_cap_basic, close, volume, sector
    """
    _, df = (
        Query()
        .set_markets("turkey")
        .select(
            "name", "close", "volume", "market_cap_basic",
            "sector", "industry", "description",
            "earnings_per_share_basic_ttm",
            "price_earnings_ttm",
            "price_book_ratio",
            "total_shares_outstanding",
        )
        .limit(limit)
        .get_scanner_data()
    )

    # Ticker temizleme: BIST:AKBNK → AKBNK
    df["ticker"] = df["ticker"].str.replace(r"^.*:", "", regex=True)

    # Filtrele: emtia sertifikaları ve niş kayıpları çıkar
    df = df[~df["ticker"].isin(NIS_KAYIPLAR | {"ALTIN", "GUMUS"})]
    df = df.reset_index(drop=True)

    print(f"BIST evreni: {len(df)} hisse")
    return df


# Test
# universe_df = get_bist_universe()
# universe_df.head()
print("BIST evren modülü hazır.")

In [ ]:
# ── borsapy — Sektör ve Temettü ───────────────────────────────────────────

import borsapy as bp


def get_sector_dividend(ticker: str) -> dict:
    """
    borsapy üzerinden sektör ve temettü bilgisi çeker.
    Sadece bu iki amaçla kullanılır — bilanço/fiyat için değil.
    """
    result = {"sector": None, "industry": None, "dividend_yield": None}
    try:
        info = bp.Ticker(ticker).info
        result["sector"]         = info.get("sector")
        result["industry"]       = info.get("industry")
        result["dividend_yield"] = info.get("dividendYield")
    except Exception as e:
        pass  # borsapy bazen hata verir; sorun değil
    return result


def classify_sector_group(ticker: str, tv_sector: str | None = None) -> str:
    """
    Hisseyi 5 sektör grubundan birine atar:
    Sinai | Finansal | NAV | Emtia | Altyapı

    Öncelik sırası: TICKER_OVERRIDE > TV sektör > borsapy sektör
    """
    if ticker in TICKER_OVERRIDE:
        raw = TICKER_OVERRIDE[ticker]
    else:
        raw = tv_sector or ""

    r = _tr_norm(raw)

    # Finansal
    if any(k in r for k in ["bank", "banka", "sigorta", "insurance",
                              "finansal", "financial"]):
        return "Finansal"

    # NAV
    if any(k in r for k in ["holding", "yatirim ortakligi", "gayrimenkul",
                              "real estate", "gyo", "investment"]):
        return "NAV"

    # Emtia
    if any(k in r for k in ["rafineri", "refinery", "madencilik", "mining",
                              "demir", "steel", "celik", "elektrik", "electric",
                              "dogalgaz", "natural gas", "enerji", "energy"]):
        return "Emtia"

    # Altyapı
    if any(k in r for k in ["telekom", "telecom", "havayolu", "airline",
                              "liman", "port", "lojistik", "logistics",
                              "denizcilik", "maritime"]):
        return "Altyapı"

    return "Sinai"  # varsayılan


def determine_financial_group(sector_group: str, ticker: str) -> str:
    """Sektör grubuna göre isyatirim financialGroup parametresi döner."""
    if sector_group == "Finansal":
        if ticker in {"TURSG", "ANSGR", "ANHYT", "AKGRT"}:
            return "UFRS_K"
        return "UFRS"
    return "XI_29"


print("borsapy sektör/temettü modülü hazır.")

In [ ]:
# ── tvDatafeed — Fiyat, Teknik, Makro ─────────────────────────────────────

from tvDatafeed import TvDatafeed, Interval
import ta

TV = TvDatafeed()   # anonim erişim

MACRO_SYMBOLS = {
    "USDTRY" : ("USDTRY",  "FX_IDC"),
    "EURTRY" : ("EURTRY",  "FX_IDC"),
    "DXY"    : ("DXY",     "TVC"),
    "TR10Y"  : ("TR10Y",   "TVC"),
    "US10Y"  : ("US10Y",   "TVC"),
    "BRENT"  : ("UKOIL",   "OANDA"),
    "GOLD"   : ("XAUUSD",  "OANDA"),
    "XU100"  : ("XU100",   "BIST"),
    "XBANK"  : ("XBANK",   "BIST"),
}


def get_price_data(
    ticker: str,
    exchange: str = "BIST",
    n_bars: int = 260,
    interval: Interval = Interval.in_daily,
) -> pd.DataFrame:
    """
    tvDatafeed üzerinden OHLCV verisi çeker.
    """
    try:
        df = TV.get_hist(
            symbol=ticker, exchange=exchange,
            interval=interval, n_bars=n_bars
        )
        if df is None or df.empty:
            return pd.DataFrame()
        return df
    except Exception as e:
        print(f"  [WARN] {ticker} fiyat hatası: {e}")
        return pd.DataFrame()


def compute_technical_indicators(df: pd.DataFrame) -> dict:
    """
    Temel teknik göstergeler hesaplar:
    RSI14, MA50, MA200, hacim_ort, momentum, fiyat
    """
    if df.empty or len(df) < 20:
        return {}

    close = df["close"]

    rsi   = ta.momentum.RSIIndicator(close, window=14).rsi()
    ma50  = close.rolling(50).mean()
    ma200 = close.rolling(200).mean()

    last_close   = float(close.iloc[-1])
    last_rsi     = float(rsi.iloc[-1]) if not rsi.empty else np.nan
    last_ma50    = float(ma50.iloc[-1]) if not ma50.empty else np.nan
    last_ma200   = float(ma200.iloc[-1]) if not ma200.empty else np.nan
    avg_vol_20   = float(df["volume"].tail(20).mean())
    last_vol     = float(df["volume"].iloc[-1])

    # 1 aylık momentum
    mom_1m = safe_div(last_close - float(close.iloc[-22]), float(close.iloc[-22]))
    # 3 aylık momentum
    mom_3m = safe_div(last_close - float(close.iloc[-66]), float(close.iloc[-66])) if len(df) >= 66 else np.nan

    return {
        "price"        : last_close,
        "rsi14"        : last_rsi,
        "ma50"         : last_ma50,
        "ma200"        : last_ma200,
        "above_ma200"  : last_close > last_ma200 if not np.isnan(last_ma200) else False,
        "above_ma50"   : last_close > last_ma50  if not np.isnan(last_ma50)  else False,
        "vol_ratio"    : safe_div(last_vol, avg_vol_20),  # hacim / 20g ort
        "mom_1m"       : mom_1m,
        "mom_3m"       : mom_3m,
    }


def get_macro_snapshot() -> dict:
    """
    Makro göstergeler için anlık değerler.
    """
    snapshot = {}
    for name, (sym, exch) in MACRO_SYMBOLS.items():
        df = get_price_data(sym, exchange=exch, n_bars=5)
        if not df.empty:
            snapshot[name] = float(df["close"].iloc[-1])
        else:
            snapshot[name] = np.nan
    return snapshot


print("tvDatafeed teknik modülü hazır.")

In [ ]:
# ── Finansal Metrikler — Altman Z & Piotroski F-Score ─────────────────────

def calc_altman_z_sinai(
    df: pd.DataFrame, market_cap: float, price: float
) -> float:
    """
    Altman Z-Score (sınai şirketler).
    Z = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5
    TR eşiği: 1.0 (klasik 1.8 değil)
    """
    items = ITEMS_SINAI

    varlik     = get_stock_value(df, items["varlik"])
    donen      = get_stock_value(df, items["donen_varlik"])
    kv_yuk     = get_stock_value(df, items["kv_yukum"])
    ozkaynak   = get_stock_value(df, items["ozkaynak"])
    hasilat    = get_ttm_value(df, items["hasilat"])
    faaliyet_k = get_ttm_value(df, items["faaliyet_kar"])
    net_kar    = get_ttm_value(df, items["net_kar"])
    fin_borc   = (
        get_stock_value(df, items["fin_borc_kv"]) +
        get_stock_value(df, items["fin_borc_uv"])
    )

    if any(np.isnan(v) for v in [varlik, donen, kv_yuk, ozkaynak]):
        return np.nan

    calisma_sermayes = donen - kv_yuk
    X1 = safe_div(calisma_sermayes, varlik)
    X2 = safe_div(ozkaynak, varlik)         # dağıtılmamış kâr proxy
    X3 = safe_div(faaliyet_k, varlik)
    X4 = safe_div(market_cap, fin_borc if not np.isnan(fin_borc) and fin_borc > 0 else 1)
    X5 = safe_div(hasilat, varlik)

    z = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5
    return round(z, 3)


def calc_piotroski_f(
    df: pd.DataFrame,
    prev_df: pd.DataFrame | None = None
) -> int:
    """
    Piotroski F-Score (9 puan, eşik=4).
    prev_df: geçen yıl bilanço (trend için). Yoksa kısmi hesap.
    """
    items = ITEMS_SINAI
    score = 0

    varlik   = get_stock_value(df, items["varlik"])
    ozkaynak = get_stock_value(df, items["ozkaynak"])
    donen    = get_stock_value(df, items["donen_varlik"])
    kv_yuk   = get_stock_value(df, items["kv_yukum"])
    net_kar  = get_ttm_value(df, items["net_kar"])
    cfo      = get_ttm_value(df, items["cfo"])
    borc_kv  = get_stock_value(df, items["fin_borc_kv"])
    borc_uv  = get_stock_value(df, items["fin_borc_uv"])

    # F1 — ROA pozitif
    roa = safe_div(net_kar, varlik)
    if not np.isnan(roa) and roa > 0:
        score += 1

    # F2 — CFO pozitif
    if not np.isnan(cfo) and cfo > 0:
        score += 1

    # F3 — CFO > net kâr (tahakkuk kalitesi)
    if not np.isnan(cfo) and not np.isnan(net_kar) and cfo > net_kar:
        score += 1

    # F4 — Cari oran iyileşmesi (mevcut yıl)
    cur_ratio = safe_div(donen, kv_yuk)
    if not np.isnan(cur_ratio) and cur_ratio > 1:
        score += 1

    # F5 — Borç/Varlık düşüşü (önceki yıl verisi gerekir)
    if prev_df is not None and not prev_df.empty:
        prev_varlik  = get_stock_value(prev_df, items["varlik"])
        prev_borc    = get_stock_value(prev_df, items["fin_borc_kv"]) + \
                       get_stock_value(prev_df, items["fin_borc_uv"])
        cur_borc = borc_kv + borc_uv
        if not any(np.isnan(x) for x in [prev_varlik, prev_borc, varlik, cur_borc]):
            if safe_div(cur_borc, varlik) < safe_div(prev_borc, prev_varlik):
                score += 1
    else:
        score += 0.5  # veri eksik, nötr

    # F6 — Yeni hisse çıkarılmamış (basit proxy: skor her zaman 0 — eksik veri)
    # F7 — Brüt marj iyileşmesi
    brut_kar = get_ttm_value(df, items["brut_kar"])
    hasilat  = get_ttm_value(df, items["hasilat"])
    gm = safe_div(brut_kar, hasilat)
    if not np.isnan(gm) and gm > 0.1:
        score += 1

    # F8 — Varlık devir hızı pozitif
    ato = safe_div(hasilat, varlik)
    if not np.isnan(ato) and ato > 0:
        score += 1

    return int(round(score))


print("Altman Z ve F-Score modülleri hazır.")

In [ ]:
# ── Değerleme Metotları ────────────────────────────────────────────────────

# ── 1) DCF (Sınai) ────────────────────────────────────────────────────────
def dcf_valuation(
    fcf_ttm: float,
    growth_rate: float = 0.15,
    terminal_growth: float = 0.05,
    wacc: float = WACC_DEFAULT,
    years: int = 10,
) -> float:
    """Basit DCF — FCF üzerinden içsel değer."""
    if np.isnan(fcf_ttm) or fcf_ttm <= 0:
        return np.nan
    pv = 0.0
    cf = fcf_ttm
    for t in range(1, years + 1):
        cf *= (1 + growth_rate)
        pv += cf / (1 + wacc) ** t
    tv = cf * (1 + terminal_growth) / (wacc - terminal_growth)
    tv_pv = tv / (1 + wacc) ** years
    return pv + tv_pv


# ── 2) Reverse DCF ────────────────────────────────────────────────────────
def reverse_dcf(
    market_cap: float,
    fcf_ttm: float,
    wacc: float = WACC_DEFAULT,
    terminal_growth: float = 0.05,
    years: int = 10,
) -> float:
    """
    Mevcut fiyatın ima ettiği büyüme oranını döner.
    Numerik çözüm (binary search).
    """
    if any(np.isnan(v) for v in [market_cap, fcf_ttm]) or fcf_ttm <= 0:
        return np.nan

    def _dcf(g):
        return dcf_valuation(fcf_ttm, g, terminal_growth, wacc, years)

    lo, hi = -0.20, 1.00
    for _ in range(50):
        mid = (lo + hi) / 2
        if _dcf(mid) < market_cap:
            lo = mid
        else:
            hi = mid
    return round((lo + hi) / 2, 4)


# ── 3) EV/EBITDA (Sınai, Emtia) ──────────────────────────────────────────
def ev_ebitda_valuation(
    ebitda: float,
    net_debt: float,
    target_multiple: float = 7.0,
) -> float:
    """EV/EBITDA → hedef piyasa değeri."""
    if np.isnan(ebitda) or ebitda <= 0:
        return np.nan
    ev = ebitda * target_multiple
    equity_val = ev - (net_debt if not np.isnan(net_debt) else 0)
    return max(equity_val, 0.0)


# ── 4) Justified P/B — Gordon Büyüme (Finansal) ──────────────────────────
def justified_pb(
    roe: float,
    cost_of_equity: float = 0.30,
    growth: float = 0.08,
) -> float:
    """
    Justified P/B = (ROE - g) / (CoE - g)
    TR için CoE genellikle %28-35.
    """
    if np.isnan(roe) or cost_of_equity <= growth:
        return np.nan
    return (roe - growth) / (cost_of_equity - growth)


# ── 5) NAV İskontosu (Holding/GYO) ───────────────────────────────────────
def nav_discount(price: float, nav_per_share: float) -> float:
    """NAV iskontosu: negatif = iskontolu, pozitif = primli."""
    if np.isnan(nav_per_share) or nav_per_share <= 0:
        return np.nan
    return (price - nav_per_share) / nav_per_share


# ── 6) EV/FCF (Altyapı) ──────────────────────────────────────────────────
def ev_fcf_valuation(
    fcf: float, net_debt: float, target_multiple: float = 10.0
) -> float:
    if np.isnan(fcf) or fcf <= 0:
        return np.nan
    return fcf * target_multiple - (net_debt if not np.isnan(net_debt) else 0)


print("Değerleme modülleri hazır.")

In [ ]:
# ── Trend Veto Kontrolü ───────────────────────────────────────────────────

def check_trend_veto(
    ticker: str,
    sector_group: str,
    sektör_detay: str,
    df: pd.DataFrame,
) -> tuple[bool, str]:
    """
    Trend bozulma veto kontrolü.
    Dönüş: (veto_var_mi, açıklama)
    """
    # Veto kapalı sektörler
    if sektör_detay in VETO_OFF_SECTORS or sector_group in ("Emtia", "Altyapı"):
        return False, "veto_kapali"

    items = ITEMS_SINAI if sector_group != "Finansal" else ITEMS_BANKA

    kar_key     = "net_kar"
    hasilat_key = "hasilat" if sector_group != "Finansal" else "faiz_geliri"

    cur_kar    = get_ttm_value(df, items[kar_key])
    prev_kar   = get_prev_year_value(df, items[kar_key])

    cur_has    = get_ttm_value(df, items.get(hasilat_key, items[kar_key]))
    prev_has   = get_prev_year_value(df, items.get(hasilat_key, items[kar_key]))

    profit_threshold = BANK_PROFIT_DROP if sector_group == "Finansal" else PROFIT_DROP_VETO

    # Kâr veto
    if not any(np.isnan(v) for v in [cur_kar, prev_kar]) and prev_kar > 0:
        chg = (cur_kar - prev_kar) / abs(prev_kar)
        if chg < profit_threshold:
            return True, f"kar_dususu_{chg:.1%}"

    # Reel hasılat veto (sadece sınai)
    if sector_group == "Sinai":
        if not any(np.isnan(v) for v in [cur_has, prev_has]) and prev_has > 0:
            nominal_chg = (cur_has - prev_has) / abs(prev_has)
            reel_chg    = nominal_chg - TUFE   # reel büyüme = nominal - enflasyon
            if reel_chg < REVENUE_DROP_VETO:
                return True, f"reel_hasilat_dususu_{reel_chg:.1%}"

    return False, "ok"


print("Trend veto modülü hazır.")

In [ ]:
# ── Tek Hisse Tam Analiz ──────────────────────────────────────────────────

def analyze_ticker(
    ticker: str,
    tv_sector: str | None = None,
    market_cap: float = np.nan,
    price: float = np.nan,
) -> dict:
    """
    Tek hisse için tam temel + teknik analiz.
    Dönüş: skor, değerleme, veto, teknik göstergeler.
    """
    result = {
        "ticker"       : ticker,
        "sector_group" : None,
        "price"        : price,
        "market_cap"   : market_cap,
        # Temel
        "hasilat_ttm"  : np.nan,
        "net_kar_ttm"  : np.nan,
        "ebitda_ttm"   : np.nan,
        "fcf_ttm"      : np.nan,
        "ozkaynak"     : np.nan,
        "net_borc"     : np.nan,
        "roe"          : np.nan,
        "pe"           : np.nan,
        "pb"           : np.nan,
        "ev_ebitda"    : np.nan,
        # Kalite
        "altman_z"     : np.nan,
        "f_score"      : np.nan,
        # Değerleme
        "dcf_value"    : np.nan,
        "implied_growth": np.nan,
        "margin_of_safety": np.nan,
        # Teknik
        "rsi14"        : np.nan,
        "above_ma200"  : False,
        "mom_1m"       : np.nan,
        "mom_3m"       : np.nan,
        # Veto
        "veto"         : False,
        "veto_reason"  : "",
        # Özet
        "signal"       : "IZLE",
        "score"        : 0,
    }

    try:
        # 1) Sektör sınıflandırması
        sector_group   = classify_sector_group(ticker, tv_sector)
        fin_group      = determine_financial_group(sector_group, ticker)
        result["sector_group"] = sector_group

        # 2) Bilanço verisi
        df = get_balance_sheet(ticker, fin_group)
        if df.empty:
            result["signal"] = "VERI_YOK"
            return result

        items = ITEMS_BANKA if sector_group == "Finansal" else ITEMS_SINAI

        # 3) Temel kalemleri hesapla
        hasilat  = get_ttm_value(df, items.get("hasilat", items.get("faiz_geliri", [])))
        net_kar  = get_ttm_value(df, items["net_kar"])
        ozkaynak = get_stock_value(df, items["ozkaynak"])
        varlik   = get_stock_value(df, items["varlik"])

        result.update({
            "hasilat_ttm" : hasilat,
            "net_kar_ttm" : net_kar,
            "ozkaynak"    : ozkaynak,
        })

        # FCF ve EBITDA (sınai/altyapı)
        if sector_group != "Finansal":
            cfo      = get_ttm_value(df, ITEMS_SINAI["cfo"])
            capex    = get_ttm_value(df, ITEMS_SINAI["capex"])
            amor     = get_ttm_value(df, ITEMS_SINAI["amortisman"])
            faiz_g   = get_ttm_value(df, ITEMS_SINAI["faiz_gideri"])
            f_kar    = get_ttm_value(df, ITEMS_SINAI["faaliyet_kar"])
            nakit    = get_stock_value(df, ITEMS_SINAI["nakit"])
            borc_kv  = get_stock_value(df, ITEMS_SINAI["fin_borc_kv"])
            borc_uv  = get_stock_value(df, ITEMS_SINAI["fin_borc_uv"])

            fcf      = cfo - abs(capex) if not any(np.isnan(v) for v in [cfo, capex]) else np.nan
            ebitda   = f_kar + amor if not any(np.isnan(v) for v in [f_kar, amor]) else np.nan
            net_borc = (borc_kv + borc_uv - nakit
                        if not any(np.isnan(v) for v in [borc_kv, borc_uv, nakit])
                        else np.nan)

            result.update({"fcf_ttm": fcf, "ebitda_ttm": ebitda, "net_borc": net_borc})

        # 4) Oranlar
        if not np.isnan(market_cap) and not np.isnan(net_kar) and net_kar > 0:
            result["pe"] = round(safe_div(market_cap, net_kar), 2)
        if not np.isnan(market_cap) and not np.isnan(ozkaynak) and ozkaynak > 0:
            result["pb"] = round(safe_div(market_cap, ozkaynak), 2)
        if not np.isnan(result["ebitda_ttm"]) and result["ebitda_ttm"] > 0:
            ev = market_cap + result.get("net_borc", 0)
            result["ev_ebitda"] = round(safe_div(ev, result["ebitda_ttm"]), 2)
        if not np.isnan(net_kar) and not np.isnan(ozkaynak) and ozkaynak > 0:
            result["roe"] = round(safe_div(net_kar, ozkaynak), 4)

        # 5) Kalite skoru
        if sector_group == "Sinai":
            result["altman_z"] = calc_altman_z_sinai(df, market_cap, price)
            result["f_score"]  = calc_piotroski_f(df)

        # 6) Değerleme
        if sector_group in ("Sinai",):
            dcf_val = dcf_valuation(result["fcf_ttm"])
            result["dcf_value"] = dcf_val
            if not np.isnan(dcf_val) and not np.isnan(market_cap) and market_cap > 0:
                result["margin_of_safety"] = round((dcf_val - market_cap) / market_cap, 4)
            result["implied_growth"] = reverse_dcf(market_cap, result["fcf_ttm"])

        elif sector_group == "Finansal":
            jPB = justified_pb(result["roe"] or 0)
            if not np.isnan(jPB) and not np.isnan(result["pb"]):
                result["margin_of_safety"] = round((jPB - result["pb"]) / jPB, 4) if jPB > 0 else np.nan

        elif sector_group == "Emtia":
            ev_val = ev_ebitda_valuation(result["ebitda_ttm"], result.get("net_borc", 0), target_multiple=6.0)
            if not np.isnan(ev_val) and not np.isnan(market_cap) and market_cap > 0:
                result["margin_of_safety"] = round((ev_val - market_cap) / market_cap, 4)

        elif sector_group == "Altyapı":
            ev_fcf_val = ev_fcf_valuation(result["fcf_ttm"], result.get("net_borc", 0), target_multiple=12.0)
            if not np.isnan(ev_fcf_val) and not np.isnan(market_cap) and market_cap > 0:
                result["margin_of_safety"] = round((ev_fcf_val - market_cap) / market_cap, 4)

        # 7) Teknik göstergeler
        price_df = get_price_data(ticker)
        if not price_df.empty:
            tech = compute_technical_indicators(price_df)
            result.update({
                "price"       : tech.get("price", price),
                "rsi14"       : tech.get("rsi14"),
                "above_ma200" : tech.get("above_ma200", False),
                "mom_1m"      : tech.get("mom_1m"),
                "mom_3m"      : tech.get("mom_3m"),
            })

        # 8) Trend veto
        sektör_detay = TICKER_OVERRIDE.get(ticker, tv_sector or "")
        veto, reason = check_trend_veto(ticker, sector_group, sektör_detay, df)
        result["veto"]        = veto
        result["veto_reason"] = reason

        # 9) Bileşik skor (0-100)
        score = 0
        if not np.isnan(result["margin_of_safety"]):
            score += min(30, max(0, int(result["margin_of_safety"] * 50)))
        if not np.isnan(result["f_score"]):
            score += int(result["f_score"]) * 3   # maks 27
        if not np.isnan(result["altman_z"]):
            if result["altman_z"] > 2.0:  score += 15
            elif result["altman_z"] > 1.0: score += 7
        if result.get("above_ma200"): score += 10
        if not np.isnan(result["rsi14"]):
            if 40 <= result["rsi14"] <= 70: score += 8
        result["score"] = min(100, score)

        # 10) Sinyal
        if veto:
            result["signal"] = "VETO"
        elif score >= 65:
            result["signal"] = "AL"
        elif score >= 40:
            result["signal"] = "IZLE"
        else:
            result["signal"] = "KAC"

    except Exception as e:
        result["signal"]     = "HATA"
        result["veto_reason"] = str(e)

    return result


print("Tek hisse analiz modülü hazır.")

In [ ]:
# ── Banka Özel Analiz ─────────────────────────────────────────────────────

def analyze_bank(ticker: str, market_cap: float, price: float) -> dict:
    """
    Bankalar için özel metrikler:
    - Leverage ratio (özkaynak/varlık) ≥ %5.5
    - NPL ≤ %8
    - NIM (net faiz marjı)
    - ROE
    - Justified P/B
    """
    base = analyze_ticker(ticker, "Bankacılık", market_cap, price)

    try:
        df = get_balance_sheet(ticker, "UFRS")
        if df.empty:
            return base

        varlik    = get_stock_value(df, ITEMS_BANKA["varlik"])
        ozkaynak  = get_stock_value(df, ITEMS_BANKA["ozkaynak"])
        krediler  = get_stock_value(df, ITEMS_BANKA["krediler"])
        takipteki = get_stock_value(df, ITEMS_BANKA["takipteki"])
        faiz_gel  = get_ttm_value(df, ITEMS_BANKA["faiz_geliri"])
        faiz_gid  = get_ttm_value(df, ITEMS_BANKA["faiz_gideri"])

        leverage  = safe_div(ozkaynak, varlik)
        npl_ratio = safe_div(abs(takipteki), abs(krediler)) if not np.isnan(takipteki) else np.nan
        nim       = safe_div(faiz_gel - abs(faiz_gid), varlik) if not any(np.isnan(v) for v in [faiz_gel, faiz_gid]) else np.nan

        base.update({
            "leverage"   : round(leverage, 4)  if not np.isnan(leverage)  else np.nan,
            "npl_ratio"  : round(npl_ratio, 4) if not np.isnan(npl_ratio) else np.nan,
            "nim"        : round(nim, 4)        if not np.isnan(nim)       else np.nan,
        })

        # CAR veto
        if not np.isnan(leverage) and leverage < CAR_MIN:
            base["veto"]        = True
            base["veto_reason"] = f"dusuk_leverage_{leverage:.1%}"
            base["signal"]      = "VETO"

        # NPL veto
        if not np.isnan(npl_ratio) and npl_ratio > NPL_MAX:
            base["veto"]        = True
            base["veto_reason"] = f"yuksek_npl_{npl_ratio:.1%}"
            base["signal"]      = "VETO"

    except Exception as e:
        base["veto_reason"] += f" | banka_analiz_hatasi: {e}"

    return base


print("Banka analiz modülü hazır.")

In [ ]:
# ── Ana Tarama Pipeline ────────────────────────────────────────────────────

from tqdm.notebook import tqdm


def run_full_scan(
    tickers: list[str] | None = None,
    min_market_cap: float = 1e9,    # min 1 milyar TL
    max_workers: int = 1,           # sıralı (API rate limit için)
    delay_sec: float = 0.5,
) -> pd.DataFrame:
    """
    Tüm BIST evrenini (veya verilen liste) tara.
    """
    # 1) Evren listesi
    if tickers is None:
        universe = get_bist_universe()
        universe = universe[universe["market_cap_basic"] >= min_market_cap]
        tickers_list = universe["ticker"].tolist()
        tv_sectors   = dict(zip(universe["ticker"], universe["sector"]))
        mkt_caps     = dict(zip(universe["ticker"], universe["market_cap_basic"]))
        prices       = dict(zip(universe["ticker"], universe["close"]))
    else:
        tickers_list = tickers
        tv_sectors   = {}
        mkt_caps     = {}
        prices       = {}

    print(f"Taranacak hisse sayısı: {len(tickers_list)}")

    results = []
    for tk in tqdm(tickers_list, desc="Tarama"):
        if tk in NIS_KAYIPLAR:
            continue
        try:
            sector_group = classify_sector_group(tk, tv_sectors.get(tk))
            if sector_group == "Finansal":
                r = analyze_bank(tk, mkt_caps.get(tk, np.nan), prices.get(tk, np.nan))
            else:
                r = analyze_ticker(tk, tv_sectors.get(tk), mkt_caps.get(tk, np.nan), prices.get(tk, np.nan))
            results.append(r)
        except Exception as e:
            results.append({"ticker": tk, "signal": "HATA", "veto_reason": str(e), "score": 0})
        time.sleep(delay_sec)

    df_results = pd.DataFrame(results)

    # Sıralama: sinyal önce, skor sonra
    signal_order = {"AL": 0, "IZLE": 1, "KAC": 2, "VETO": 3, "VERI_YOK": 4, "HATA": 5}
    df_results["_sig_ord"] = df_results["signal"].map(signal_order).fillna(9)
    df_results = df_results.sort_values(["_sig_ord", "score"], ascending=[True, False])
    df_results = df_results.drop(columns=["_sig_ord"]).reset_index(drop=True)

    return df_results


print("Ana tarama pipeline hazır.")

In [ ]:
# ── Makro Analiz ──────────────────────────────────────────────────────────

def print_macro_snapshot():
    """Makro göstergelerin özeti."""
    snap = get_macro_snapshot()
    print("\n═══ MAKRO SNAPSHOT ═══")
    for k, v in snap.items():
        print(f"  {k:12s}: {v:.4f}" if not np.isnan(v) else f"  {k:12s}: -")
    print("═══════════════════════\n")
    return snap


# print_macro_snapshot()
print("Makro analiz modülü hazır.")

In [ ]:
# ── Excel Export ─────────────────────────────────────────────────────────

import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

SIGNAL_COLORS = {
    "AL"      : "00B050",  # yeşil
    "IZLE"    : "FFC000",  # sarı
    "KAC"     : "FF7043",  # turuncu
    "VETO"    : "C00000",  # kırmızı
    "VERI_YOK": "BDBDBD",  # gri
    "HATA"    : "BDBDBD",
}


def export_to_excel(df: pd.DataFrame, filename: str | None = None) -> str:
    """
    Sonuçları renkli Excel'e yazar.
    """
    if filename is None:
        ts = datetime.now().strftime("%Y%m%d_%H%M")
        filename = f"{DRIVE_ROOT}/raporlar/bist_analiz_{ts}.xlsx"

    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="Sonuçlar")

        ws = writer.sheets["Sonuçlar"]

        # Başlık satırını kalın yap
        for cell in ws[1]:
            cell.font      = Font(bold=True, color="FFFFFF")
            cell.fill      = PatternFill("solid", fgColor="37474F")
            cell.alignment = Alignment(horizontal="center")

        # Sinyal kolonunu boya
        signal_col_idx = None
        for col_idx, cell in enumerate(ws[1], 1):
            if cell.value == "signal":
                signal_col_idx = col_idx
                break

        if signal_col_idx:
            for row in ws.iter_rows(min_row=2, min_col=signal_col_idx, max_col=signal_col_idx):
                for cell in row:
                    color = SIGNAL_COLORS.get(str(cell.value), "FFFFFF")
                    cell.fill = PatternFill("solid", fgColor=color)
                    cell.font = Font(bold=True, color="FFFFFF")

        # Kolon genişliklerini otomatik ayarla
        for col in ws.columns:
            max_len = max((len(str(c.value)) for c in col if c.value), default=10)
            ws.column_dimensions[get_column_letter(col[0].column)].width = min(max_len + 2, 30)

    print(f"Excel kaydedildi: {filename}")
    return filename


print("Excel export modülü hazır.")

In [ ]:
# ── Özet Rapor ────────────────────────────────────────────────────────────

def print_summary(df: pd.DataFrame):
    """Tarama sonuçlarının özet tablosunu yazdırır."""
    total = len(df)
    counts = df["signal"].value_counts()

    print("\n" + "═" * 55)
    print("  BIST TEMEL ANALİZ — TARAMA ÖZETI")
    print("═" * 55)
    print(f"  Toplam taranmış hisse : {total}")
    for sig in ["AL", "IZLE", "VETO", "KAC", "VERI_YOK", "HATA"]:
        n = counts.get(sig, 0)
        bar = "█" * (n * 20 // max(total, 1))
        print(f"  {sig:10s} : {n:4d}  {bar}")
    print("═" * 55)

    print("\n  ── AL SİNYALİ ALANLAR ──")
    al_df = df[df["signal"] == "AL"][["ticker", "sector_group", "score",
                                       "pe", "pb", "margin_of_safety",
                                       "rsi14", "above_ma200"]]
    if al_df.empty:
        print("  (yok)")
    else:
        print(al_df.to_string(index=False))
    print()


print("Özet rapor modülü hazır.")

## Çalıştırma

Aşağıdaki hücreyi çalıştırarak tam BIST taramasını başlatabilirsiniz.

- `min_market_cap`: Minimum piyasa değeri filtresi (TL)
- `delay_sec`: API rate limit için bekleme süresi (saniye)
- Sonuçlar otomatik olarak Excel'e kaydedilir.

**Tek hisse testi için:**
```python
r = analyze_ticker("AKBNK", market_cap=250e9, price=45.0)
```

In [ ]:
# ── ÇALIŞTIR ─────────────────────────────────────────────────────────────

# Makro görünüm
macro = print_macro_snapshot()

# Tam tarama
# Küçük test için: tickers=["AKBNK", "GARAN", "THYAO", "SISE", "KCHOL"]
results_df = run_full_scan(
    tickers      = None,         # None → tüm BIST evreni
    min_market_cap = 1e9,        # 1 milyar TL alt sınır
    delay_sec    = 0.5,
)

# Özet
print_summary(results_df)

# Excel'e yaz
excel_path = export_to_excel(results_df)

# Sonuçları göster
results_df.head(20)